# Steel scratch triage · Independent TreeSHAP audit

Run this companion notebook from `final/steel-scratch-triage/` with the project's pinned dependencies plus `shap==0.51.0` and `ipykernel`. It reads the cleaned run's saved results. To regenerate those results, run `python analysis.py` first.

Additivity checks the total contribution. Two attribution errors can cancel and still pass. This audit refits the selected tree model on the same training rows, checks its raw scores against the saved explanations, then compares the base value and each feature contribution with `shap.TreeExplainer`. Both methods use tree-path-dependent attributions in raw log-odds. The audit covers every retained validation row and reports the slide's local example separately.

## What this audit verifies

**It verifies the explanation, and it changes none of the final metrics.** The check belongs in the **interpretability and validation** part of the project, after the tree model is trained. It is separate from preprocessing, model training, calibration, and threshold selection.

The model gives each fault a scratch score. TreeSHAP explains that score by assigning a contribution to each feature. The original additivity check confirmed that the contributions **sum to the score** once the base value is included. If plate thickness got 0.2 too much credit and brightness got 0.2 too little, the total would still match. The independent check compares **each feature’s contribution and the base value** against the SHAP library. Both implementations agreed to floating-point precision on all **387 retained validation records** after cleaning.

| Project step | What changed |
| --- | --- |
| **Preprocessing** | Dropping the two brightness rows changed which records were evaluated. That was a separate change. |
| **Model results** | The TreeSHAP check leaves AUC, probabilities, the **0.1626 cutoff**, and the **112 of 115** holdout result unchanged. |
| **Interpretability** | The check verifies the calculations behind the **global feature ranking** and **local missed-scratch explanation**. |

**For the presentation:** “We cross-checked our TreeSHAP implementation against the SHAP library and confirmed that the individual feature contributions agreed to numerical precision.”

The audit verifies the calculations that explain how the model scored a fault. Establishing the physical causes of scratches would require separate evidence.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from analysis import load_data, make_splits

data = Path('data/faults.csv')
results = Path('results')
x, y, subtype, audit = load_data(data)
splits = make_splits(y, audit['excluded_brightness_rows'])
s = json.loads((results / 'summary.json').read_text())
if s['audit'] != audit:
    raise AssertionError('Saved results and current data differ. Run analysis.py first.')
print(f"{audit['rows']:,} retained faults, including {audit['positive']:,} scratches.")
print(f"Excluded source rows: {[r['row_id'] for r in audit['excluded_brightness_rows']]}")


1,939 retained faults, including 579 scratches.
Excluded source rows: [518, 591]


In [2]:
import numpy as np
import shap as shap_library
import sklearn
from numpy.testing import assert_allclose
from analysis import candidate_models, raw_score

model_name = s['shap']['model']
params = next(row['parameters'] for row in s['models'] if row['model'] == model_name)
audit_model = candidate_models()[model_name][0].set_params(**params)
audit_model.fit(x.loc[splits['train']], y.loc[splits['train']])
validation_x = x.loc[splits['validation']]

custom = pd.read_csv(results / 'validation_tree_shap.csv', index_col='row_id')
if not custom.index.equals(validation_x.index) or not custom.columns.equals(validation_x.columns):
    raise AssertionError('Validation row IDs and feature columns must match in order.')
custom_base = s['shap']['base']
margins = raw_score(audit_model, validation_x)
tolerance = 1e-8
assert_allclose(custom_base + custom.sum(axis=1), margins, atol=tolerance, rtol=0,
                err_msg='The refit does not reproduce the analysis margins.')

reference = shap_library.TreeExplainer(
    audit_model, feature_perturbation='tree_path_dependent', model_output='raw',
)(validation_x, check_additivity=True)
reference_values = np.asarray(reference.values)
if reference_values.shape != custom.shape or list(reference.feature_names) != list(custom.columns):
    raise AssertionError('Unexpected TreeExplainer output shape or feature order.')
reference_base = np.asarray(reference.base_values).reshape(-1)
base_gap = np.max(np.abs(custom_base - reference_base))
errors = pd.DataFrame(np.abs(custom.to_numpy() - reference_values),
                      index=custom.index, columns=custom.columns)
worst_row, worst_column = np.unravel_index(errors.to_numpy().argmax(), errors.shape)
print(f'SHAP {shap_library.__version__} | scikit-learn {sklearn.__version__}')
print(f'Base values: custom {custom_base:.12f}, reference {reference_base[0]:.12f}')
print(f'Maximum base-value difference: {base_gap:.3e}')
print(f'Largest attribution difference: {errors.iat[worst_row, worst_column]:.3e} '
      f'(source row {errors.index[worst_row]}, {errors.columns[worst_column]})')
print(f'Local case (source row {s["shap"]["case_row_id"]}) maximum difference: '
      f'{errors.loc[s["shap"]["case_row_id"]].max():.3e}')

print(f'{len(errors)} retained validation rows × {errors.shape[1]} features checked.')
assert_allclose(reference_base, custom_base, atol=tolerance, rtol=0,
                err_msg='TreeSHAP base values disagree.')
assert_allclose(reference_values, custom.to_numpy(), atol=tolerance, rtol=0,
                err_msg='One or more individual TreeSHAP contributions disagree.')
reference_frame = pd.DataFrame(reference_values, index=custom.index, columns=custom.columns)
global_importance = pd.DataFrame({
    'Custom mean |SHAP|': custom.abs().mean(),
    'Reference mean |SHAP|': reference_frame.abs().mean(),
})
display(global_importance.sort_values('Custom mean |SHAP|', ascending=False).head(5).round(6))
print(f'PASS: all base values and individual contributions agree within {tolerance:g} raw-score units.')

SHAP 0.51.0 | scikit-learn 1.8.0
Base values: custom -1.813085855996, reference -1.813085855996
Maximum base-value difference: 3.775e-15
Largest attribution difference: 6.217e-15 (source row 514, Outside_X_Index)
Local case (source row 161) maximum difference: 6.661e-16
387 retained validation rows × 20 features checked.
PASS: all base values and individual contributions agree within 1e-08 raw-score units.


,Custom mean |SHAP|,Reference mean |SHAP|
Outside_X_Index,0.950448,0.950448
Steel_Plate_Thickness,0.854926,0.854926
Edges_Index,0.659518,0.659518
TypeOfSteel_A300,0.309069,0.309069
Minimum_of_Luminosity,0.295893,0.295893
